# NBA — Diagnóstico das bases e da concentração (V1)

**Somente leitura.** Não treina, não reamostra o treino, não altera parâmetros,
fontes, regras de estado ou tabelas. Importar em um **notebook separado**.

A pergunta é: a concentração vem da telemetria, de poucos clientes, da
representação, da amostra de ajuste ou da falta de memória/contexto?

Unidades que não devem ser confundidas:

- **Registro bruto:** linha da fonte, que pode ser telemetria repetida.
- **Evento preparado:** linha após profundidade/debounce; não é necessariamente
  uma transação financeira distinta.
- **Bloco de funil:** eventos consecutivos da mesma ação/funil, sem atravessar
  ambiguidades. Não equivale a uma sessão, pagamento concluído ou transação.
- **Cliente:** para medir presença e não deixar usuários muito ativos
  dominarem todas as estatísticas.

A configuração padrão analisa apenas os clientes de **treino** da base
preparada persistida. O holdout aparece no inventário, mas não é misturado
às análises exploratórias. A base preparada **não é a amostra final por
origem** recebida pelo SciPy; a seção de suporte mostra essa diferença.

Os resultados são descritivos. Frequência de uso, distribuição de Top 1 e
probabilidade de primeira passagem são objetos diferentes.

In [ ]:
from pyspark import StorageLevel
from pyspark.sql import Column, DataFrame, Window
from pyspark.sql import functions as F


DIAG_CFG = {
    "tabela_base": (
        "ctg_dsti.renato_nba.nba_sm_v22_base_treino_clean_hml"
    ),
    "tabela_config": (
        "ctg_dsti.renato_nba.nba_config_estados_v222_clean"
    ),
    "tabela_modelos": (
        "ctg_dsti.renato_nba.nba_sm_v22_modelos_clean_hml"
    ),
    "tabela_scoring": (
        "ctg_dsti.renato_nba.nba_semimarkov_v222_macroaware_long_hml"
    ),
    "fonte_raw": "ctg_dsti.renato_nba.base_jornadas_onboarding",
    "id_execucao_base": None,  # Obrigatório se houver várias execuções.
    "id_execucao_modelos": None,
    "escopo": "TREINO",  # TREINO, VALIDACAO ou TODOS; manter TREINO na EDA.
    "fuso_esperado": "Etc/UTC",  # O notebook não muda o fuso da sessão.
    "analisar_raw": False,  # Bloco opcional de leitura da fonte maior.
    "inicio_raw": None,  # Informar o início REAL usado na preparação.
    "fim_raw_exclusivo": None,  # Se None, usa ts_corte_estado da base.
    "analisar_scoring": True,
    "modulo_scoring": 1000,  # Amostra do diagnóstico, NÃO do modelo.
    "sal_scoring": "nba_eda_scoring_v1",
    "min_clientes_contexto": 30,  # Apenas exibição; não ajusta modelos.
    "max_linhas_exibir": 40,
}

ACOES_TECNICAS_DIAG = {
    "app_login", "app_habilitacao_device", "app_primeiro_acesso",
    "app_reset_senha", "app_atualizacao_cadastral",
}
ACOES_FOCO_DIAG = [
    "pix_transferencia", "pix_pagamento", "pagamentos_boleto",
]
DIAG_RESULTADOS = {}
DIAG_CACHES = []


def diag_exigir_colunas(df: DataFrame, nomes: set, origem: str) -> None:
    faltantes = nomes - set(df.columns)
    if faltantes:
        raise ValueError(f"{origem}: colunas ausentes: {sorted(faltantes)}")


def diag_tem_linhas(df: DataFrame) -> bool:
    return bool(df.limit(1).count())


def diag_ler(tabela: str) -> DataFrame:
    if not spark.catalog.tableExists(tabela):
        raise ValueError(f"Tabela/view não encontrada: {tabela}")
    return spark.table(tabela)


def diag_execucao_unica(
    df: DataFrame, identificador: str | None, nome: str
) -> tuple[DataFrame, str | None]:
    if "id_execucao" not in df.columns:
        if identificador is not None:
            raise ValueError(f"{nome}: não possui id_execucao para filtrar.")
        print(f"AVISO: {nome} não permite conferir id_execucao.")
        return df, None
    if identificador is not None:
        df = df.filter(F.col("id_execucao") == F.lit(identificador))
    ids = df.select("id_execucao").distinct().limit(3).collect()
    if len(ids) != 1 or ids[0]["id_execucao"] is None:
        raise ValueError(
            f"{nome}: selecione uma única execução não nula na configuração."
        )
    return df, str(ids[0]["id_execucao"])


def diag_mostrar(nome: str, df: DataFrame) -> None:
    DIAG_RESULTADOS[nome] = df
    print(f"\n{nome}")
    df.show(DIAG_CFG["max_linhas_exibir"], truncate=False)


def diag_cache(df: DataFrame) -> DataFrame:
    cached = df.persist(StorageLevel.MEMORY_AND_DISK)
    DIAG_CACHES.append(cached)
    return cached


def diag_dividir(numerador: Column, denominador: Column) -> Column:
    return F.when(denominador > 0, numerador / denominador)


if "spark" not in globals():
    raise RuntimeError("Execute este notebook em um ambiente com Spark.")
if DIAG_CFG["escopo"] not in {"TREINO", "VALIDACAO", "TODOS"}:
    raise ValueError("escopo deve ser TREINO, VALIDACAO ou TODOS.")
if DIAG_CFG["modulo_scoring"] < 1:
    raise ValueError("modulo_scoring deve ser >= 1.")

fuso_diag = spark.conf.get("spark.sql.session.timeZone")
print("Fuso da sessão:", fuso_diag)
if fuso_diag != DIAG_CFG["fuso_esperado"]:
    raise ValueError(
        "Fuso diferente do esperado. Confira a origem antes de ajustar "
        "fuso_esperado; não altere timestamps apenas para passar na checagem."
    )
print("Execução de diagnóstico: sem escrita permanente e sem treinamento.")

## 1. Inventário, granularidade e escopo

Lê as tabelas persistidas. Não exige variáveis do notebook de treinamento.
Interrompe se encontrar duplicidade de cliente/passo, múltiplas execuções
não selecionadas ou metadados conflitantes para um estado.

`funil` é utilizado como agrupamento de negócio. Atendimento mantém o estado
como categoria, sem transformar canal em etapa. Estados sem configuração
ficam identificados e **não são excluídos silenciosamente**.

In [ ]:
diag_base_full, DIAG_ID_BASE = diag_execucao_unica(
    diag_ler(DIAG_CFG["tabela_base"]),
    DIAG_CFG["id_execucao_base"],
    "base preparada",
)
diag_exigir_colunas(
    diag_base_full,
    {"cd_bv", "passo", "estado", "ts_estado", "destino", "dur_min",
     "tipo_censura", "elegivel_ajuste", "validacao_cliente"},
    "base preparada",
)
if not diag_tem_linhas(diag_base_full):
    raise ValueError("Base preparada vazia.")
if diag_tem_linhas(diag_base_full.filter(
    F.col("cd_bv").isNull() | F.col("passo").isNull()
    | F.col("validacao_cliente").isNull()
)):
    raise ValueError("Cliente, passo ou indicador de validação nulo.")
if diag_tem_linhas(
    diag_base_full.groupBy("cd_bv", "passo").count().filter("count > 1")
):
    raise ValueError("Cliente/passo repetido: verificar versões e duplicatas.")
if diag_tem_linhas(
    diag_base_full.groupBy("cd_bv").agg(
        F.countDistinct("validacao_cliente").alias("n_splits")
    ).filter("n_splits > 1")
):
    raise ValueError("O mesmo cliente aparece em treino e validação.")

diag_mostrar("01_inventario_treino_validacao", (
    diag_base_full.groupBy("validacao_cliente").agg(
        F.count("*").alias("n_linhas_preparadas"),
        F.countDistinct("cd_bv").alias("n_clientes"),
        F.countDistinct("estado").alias("n_estados"),
        F.min("ts_estado").alias("primeiro_ts_preparado"),
        F.max("ts_estado").alias("ultimo_ts_preparado"),
    ).orderBy("validacao_cliente")
))

if DIAG_CFG["escopo"] == "TREINO":
    diag_base = diag_base_full.filter(~F.col("validacao_cliente"))
elif DIAG_CFG["escopo"] == "VALIDACAO":
    diag_base = diag_base_full.filter(F.col("validacao_cliente"))
else:
    diag_base = diag_base_full
if not diag_tem_linhas(diag_base):
    raise ValueError("Escopo selecionado vazio.")

diag_config = diag_ler(DIAG_CFG["tabela_config"])
diag_exigir_colunas(
    diag_config, {"estado", "funil", "etapa", "tipo_estado"}, "config"
)
diag_config = diag_config.select(
    "estado", "funil", "etapa", "tipo_estado"
).distinct()
if diag_tem_linhas(diag_config.groupBy("estado").count().filter("count > 1")):
    raise ValueError("Metadados conflitantes para o mesmo estado.")

diag_base = diag_cache(
    diag_base.join(diag_config, "estado", "left")
    .withColumn("acao", F.coalesce("funil", "estado"))
    .withColumn("sem_config", F.col("tipo_estado").isNull())
    .withColumn("evento_valido", (
        F.col("estado").isNotNull() & F.col("ts_estado").isNotNull()
    ))
    .withColumn("acao_tecnica", F.col("acao").isin(
        sorted(ACOES_TECNICAS_DIAG)
    ))
)
diag_eventos = diag_base.filter(F.col("evento_valido"))
DIAG_N_CLIENTES = diag_eventos.select("cd_bv").distinct().count()
if DIAG_N_CLIENTES == 0:
    raise ValueError("Nenhum cliente com evento válido no escopo.")

diag_mostrar("01_qualidade", diag_base.agg(
    F.count("*").alias("n_linhas"),
    F.sum((~F.col("evento_valido")).cast("long")).alias("n_invalidas"),
    F.sum(F.col("sem_config").cast("long")).alias("n_sem_config"),
    F.sum(F.col("elegivel_ajuste").cast("long")).alias("n_elegiveis"),
    F.sum((F.col("dur_min") <= 0).cast("long")).alias("n_duracao_nao_pos"),
))
diag_mostrar("01_status_observacao", (
    diag_base.groupBy("tipo_censura", "elegivel_ajuste").count()
    .orderBy(F.desc("count"))
))
diag_mostrar("01_estados_sem_config", (
    diag_base.filter(F.col("sem_config")).groupBy("estado").count()
    .orderBy(F.desc("count"))
))
print("Clientes com evento observável no escopo:", DIAG_N_CLIENTES)
print("Data máxima de evento NÃO comprova completude da fonte.")

## 2. Blocos consecutivos de funil

`topo → navegacao → sucesso` do mesmo funil forma um bloco. Não há limite
de 30 minutos nem estado de silêncio. Uma barreira ambígua quebra o bloco.
**Um bloco não é uma sessão real:** repetições consecutivas do mesmo funil
em dias diferentes podem pertencer ao mesmo bloco nesta definição.

A comparação entre eventos preparados e blocos revela quanto a divisão
em etapas aumenta o volume de uma ação.

In [ ]:
w_eventos_diag = Window.partitionBy("cd_bv").orderBy("passo")
marcados_diag = (
    diag_base.withColumn("_acao_prev", F.lag("acao").over(w_eventos_diag))
    .withColumn("_valido_prev", F.lag("evento_valido").over(w_eventos_diag))
    .withColumn("_novo_bloco", F.when(
        F.col("evento_valido") & F.coalesce("_valido_prev", F.lit(False))
        & F.col("acao").eqNullSafe(F.col("_acao_prev")), 0
    ).otherwise(1))
    .withColumn("bloco_id", F.sum("_novo_bloco").over(
        w_eventos_diag.rowsBetween(Window.unboundedPreceding, 0)
    ))
)
diag_blocos_full = diag_cache(marcados_diag.groupBy(
    "cd_bv", "bloco_id", "acao"
).agg(
    F.min("passo").alias("passo_inicio"),
    F.max("passo").alias("passo_fim"),
    F.min("ts_estado").alias("ts_inicio_bloco"),
    F.max("ts_estado").alias("ts_fim_bloco"),
    F.count("*").alias("n_eventos_bloco"),
    F.countDistinct("estado").alias("n_estados_bloco"),
    F.min(F.struct("passo", "estado")).alias("_entrada_bloco"),
    F.min(F.col("evento_valido").cast("int")).alias("bloco_valido"),
    F.max(F.col("sem_config").cast("int")).alias("tem_sem_config"),
).withColumn(
    "estado_entrada", F.col("_entrada_bloco.estado")
).drop("_entrada_bloco"))
diag_blocos = diag_blocos_full.filter(F.col("bloco_valido") == 1)

diag_mostrar("02_blocos_por_funil", diag_blocos.groupBy("acao").agg(
    F.count("*").alias("n_blocos_funil"),
    F.sum("n_eventos_bloco").alias("n_eventos_preparados"),
    F.countDistinct("cd_bv").alias("n_clientes"),
    F.avg("n_eventos_bloco").alias("eventos_por_bloco"),
    F.expr("percentile_approx(n_eventos_bloco, array(0.5, 0.9, 0.99))")
    .alias("quantis_eventos_por_bloco"),
).orderBy(F.desc("n_blocos_funil")))

## 3. Presença por cliente versus participação no volume

Para uma ação `a`, o notebook calcula:

- `participacao_volume`: número de eventos/blocos da ação dividido pelo total.
- `penetracao_clientes`: clientes com ao menos uma ocorrência / clientes com
  eventos observados no escopo. **As penetrações não somam 1.**
- `participacao_media_por_cliente`: média da participação da ação dentro de
  cada cliente, incluindo zero para quem não a teve. Soma 1 entre ações.

Esta última dá o mesmo peso a cada cliente. É um diagnóstico de sensibilidade,
não uma substituição automática do estimador por evento.

In [ ]:
def diag_perfil_presenca(df: DataFrame, categoria: str) -> DataFrame:
    contagens = df.groupBy("cd_bv", categoria).count().withColumnRenamed(
        "count", "n_ocorrencias"
    )
    contagens = contagens.withColumn(
        "n_cliente", F.sum("n_ocorrencias").over(Window.partitionBy("cd_bv"))
    ).withColumn("participacao_cliente", F.col("n_ocorrencias") / F.col(
        "n_cliente"
    ))
    total = df.agg(F.count("*").alias("n_total_ocorrencias"))
    return (
        contagens.groupBy(categoria).agg(
            F.sum("n_ocorrencias").alias("n_ocorrencias"),
            F.count("*").alias("n_clientes_com_presenca"),
            F.avg("n_ocorrencias").alias("media_entre_usuarios_da_acao"),
            F.sum("participacao_cliente").alias("soma_participacoes"),
        ).crossJoin(total)
        .withColumn("participacao_volume", diag_dividir(
            F.col("n_ocorrencias"), F.col("n_total_ocorrencias")
        ))
        .withColumn("penetracao_clientes", F.col(
            "n_clientes_com_presenca"
        ) / F.lit(DIAG_N_CLIENTES))
        .withColumn("participacao_media_por_cliente", F.col(
            "soma_participacoes"
        ) / F.lit(DIAG_N_CLIENTES))
        .drop("soma_participacoes", "n_total_ocorrencias")
        .orderBy(F.desc("participacao_volume"))
    )


diag_mostrar("03_presenca_estado", diag_perfil_presenca(
    diag_eventos, "estado"
))
diag_mostrar("03_presenca_funil_eventos", diag_perfil_presenca(
    diag_eventos, "acao"
))
diag_mostrar("03_presenca_funil_blocos", diag_perfil_presenca(
    diag_blocos, "acao"
))

## 4. Concentração individual e clientes muito ativos

`HHI = soma(p_acao²)` dentro de cada cliente: próximo de 1 significa
concentração; `1/HHI` é o número efetivo de ações. Entropia em bits é outro
resumo de diversidade. Pouco histórico também produz concentração: os
relatórios separam faixas de volume e mostram a duração observada do histórico.

Pareto usa quantis aproximados de volume; empates no limiar são incluídos.
A fração real de clientes selecionada aparece no relatório.

In [ ]:
diag_freq_cliente = diag_blocos.groupBy("cd_bv", "acao").count()
diag_freq_cliente = diag_freq_cliente.withColumn(
    "n_blocos_cliente", F.sum("count").over(Window.partitionBy("cd_bv"))
).withColumn("p", F.col("count") / F.col("n_blocos_cliente"))

diag_concentracao = diag_freq_cliente.groupBy("cd_bv").agg(
    F.max("n_blocos_cliente").alias("n_blocos"),
    F.count("*").alias("n_funis"),
    F.max("p").alias("share_funil_dominante"),
    F.sum(F.col("p") * F.col("p")).alias("hhi"),
    F.sum(-F.col("p") * F.log2("p")).alias("entropia_bits"),
).withColumn("n_funis_efetivos", 1.0 / F.col("hhi"))

diag_perfil_clientes = diag_eventos.groupBy("cd_bv").agg(
    F.count("*").alias("n_eventos"),
    F.countDistinct("estado").alias("n_estados"),
    F.countDistinct(F.to_date("ts_estado")).alias("n_dias_ativos"),
    F.min("ts_estado").alias("primeira_evidencia"),
    F.max("ts_estado").alias("ultima_evidencia"),
).join(diag_concentracao, "cd_bv").withColumn(
    "amplitude_historico_dias",
    F.datediff("ultima_evidencia", "primeira_evidencia")
).withColumn(
    "faixa_volume", F.when(F.col("n_blocos") < 5, "01_1_a_4")
    .when(F.col("n_blocos") < 20, "02_5_a_19")
    .when(F.col("n_blocos") < 100, "03_20_a_99").otherwise("04_100_ou_mais")
)
diag_perfil_clientes = diag_cache(diag_perfil_clientes)

diag_mostrar("04_concentracao_por_cliente", (
    diag_perfil_clientes.groupBy("faixa_volume").agg(
        F.count("*").alias("n_clientes"),
        F.avg("n_eventos").alias("media_eventos"),
        F.avg("n_funis").alias("media_funis"),
        F.avg("n_funis_efetivos").alias("media_funis_efetivos"),
        F.avg("share_funil_dominante").alias("media_share_dominante"),
        F.avg("entropia_bits").alias("media_entropia_bits"),
        F.avg("amplitude_historico_dias").alias("media_amplitude_dias"),
        F.avg("n_dias_ativos").alias("media_dias_ativos"),
    ).orderBy("faixa_volume")
))

limiares_diag = diag_perfil_clientes.approxQuantile(
    "n_blocos", [0.90, 0.95, 0.99], 0.001
)
pareto_diag = []
for quantil, limiar in zip([0.90, 0.95, 0.99], limiares_diag):
    tabela = diag_perfil_clientes.agg(
        F.count("*").alias("n_total_clientes"),
        F.sum("n_blocos").alias("total_blocos"),
        F.sum((F.col("n_blocos") >= limiar).cast("long"))
        .alias("n_clientes_acima_limiar"),
        F.sum(F.when(F.col("n_blocos") >= limiar, F.col("n_blocos"))
              .otherwise(0)).alias("blocos_acima_limiar"),
    ).withColumn("quantil_limiar", F.lit(quantil)).withColumn(
        "limiar_blocos", F.lit(limiar)
    ).withColumn("fracao_clientes_real", F.col(
        "n_clientes_acima_limiar"
    ) / F.col("n_total_clientes")).withColumn(
        "fracao_blocos", F.col("blocos_acima_limiar") / F.col("total_blocos")
    )
    pareto_diag.append(tabela)
if pareto_diag:
    pareto_resultado = pareto_diag[0]
    for tabela in pareto_diag[1:]:
        pareto_resultado = pareto_resultado.unionByName(tabela)
    diag_mostrar("04_pareto_clientes", pareto_resultado)

## 5. Transições e censura

Aqui as frequências são calculadas **apenas nas saídas exatas observadas**.
Não são o `p_destino` estimado pelo Semi-Markov com censura. Uma frequência
calculada somente entre saídas concluídas pode diferir da probabilidade
estimada com todas as observações.

Para cada origem, compara frequência por evento com a média das frequências
individuais (peso igual entre clientes com saída daquela origem).

In [ ]:
diag_transicoes = diag_base.filter(
    F.col("elegivel_ajuste") & (F.col("tipo_censura") == "exata")
    & F.col("destino").isNotNull()
)
diag_td = diag_transicoes.groupBy("cd_bv", "estado", "destino").count()
diag_td = diag_td.withColumn("n_cd_origem", F.sum("count").over(
    Window.partitionBy("cd_bv", "estado")
)).withColumn("p_cd", F.col("count") / F.col("n_cd_origem"))
diag_origens = diag_transicoes.groupBy("estado").agg(
    F.count("*").alias("n_saidas_origem"),
    F.countDistinct("cd_bv").alias("n_clientes_origem"),
)
diag_pares = (
    diag_td.groupBy("estado", "destino").agg(
        F.sum("count").alias("n_saidas_par"),
        F.count("*").alias("n_clientes_par"),
        F.sum("p_cd").alias("soma_p_cliente"),
    ).join(diag_origens, "estado")
    .withColumn("p_evento_exato",
                F.col("n_saidas_par") / F.col("n_saidas_origem"))
    .withColumn("p_cliente_igual",
                F.col("soma_p_cliente") / F.col("n_clientes_origem"))
    .withColumn("delta_ponderacao",
                F.col("p_cliente_igual") - F.col("p_evento_exato"))
    .withColumn("rank_origem", F.row_number().over(
        Window.partitionBy("estado").orderBy(F.desc("n_saidas_par"), "destino")
    ))
)
diag_mostrar("05_transicoes_top", diag_pares.filter(
    F.col("rank_origem") <= 3
).orderBy(F.desc("n_saidas_origem"), "estado", "rank_origem"))
diag_mostrar("05_sensibilidade_peso_cliente", diag_pares.filter(
    F.col("n_clientes_par") >= DIAG_CFG["min_clientes_contexto"]
).orderBy(F.desc(F.abs("delta_ponderacao"))))

diag_mostrar("05_censura_duracao", diag_base.groupBy("estado").agg(
    F.count("*").alias("n_linhas"),
    F.avg((F.col("tipo_censura") == "direita").cast("double"))
    .alias("fracao_censurada_direita"),
    F.sum((F.col("tipo_censura") == "exata").cast("long"))
    .alias("n_saidas_exatas"),
    F.expr(
        "percentile_approx(CASE WHEN tipo_censura = 'exata' "
        "THEN dur_min END, array(0.5, 0.9, 0.99))"
    ).alias("quantis_duracao_exata_seg"),
).orderBy(F.desc("n_linhas")))

## 6. O login mistura contextos diferentes?

Cria triplas de blocos: `ação anterior → app_login → próxima ação`.
A comparação pergunta se a distribuição após login muda com a ação anterior,
separando também o estado RAW de entrada no login. Isso reduz a mistura
entre entradas em etapas diferentes, mas não controla todas as diferenças
de idade, perfil ou calendário.

Não atravessa blocos ambíguos/sem configuração. Casos sem próximo bloco
observado são reportados, **não tratados como destino negativo**. O resultado
é condicional às triplas completas, observacional e não causal.

In [ ]:
w_blocos_diag = Window.partitionBy("cd_bv").orderBy("bloco_id")
diag_triplas = (
    diag_blocos_full.withColumn(
        "acao_anterior", F.lag("acao").over(w_blocos_diag)
    )
    .withColumn("valido_anterior", F.lag("bloco_valido").over(w_blocos_diag))
    .withColumn("sem_config_anterior",
                F.lag("tem_sem_config").over(w_blocos_diag))
    .withColumn("acao_seguinte", F.lead("acao").over(w_blocos_diag))
    .withColumn("valido_seguinte", F.lead("bloco_valido").over(w_blocos_diag))
    .withColumn("sem_config_seguinte",
                F.lead("tem_sem_config").over(w_blocos_diag))
)
diag_logins = diag_triplas.filter(F.col("acao") == "app_login")
diag_logins = diag_logins.withColumn("tripla_completa", (
    (F.col("bloco_valido") == 1) & (F.col("tem_sem_config") == 0)
    & (F.col("valido_anterior") == 1) & (F.col("sem_config_anterior") == 0)
    & (F.col("valido_seguinte") == 1) & (F.col("sem_config_seguinte") == 0)
))
diag_mostrar("06_cobertura_triplas_login", diag_logins.agg(
    F.count("*").alias("n_blocos_login"),
    F.sum(F.coalesce("tripla_completa", F.lit(False)).cast("long"))
    .alias("n_triplas_completas"),
    F.countDistinct("cd_bv").alias("n_clientes_login"),
))
diag_login_completo = diag_logins.filter(F.col("tripla_completa"))
diag_login_contexto = diag_login_completo.groupBy(
    "acao_anterior", "estado_entrada", "acao_seguinte"
).agg(
    F.count("*").alias("n_triplas"),
    F.countDistinct("cd_bv").alias("n_clientes_par"),
)
diag_login_totais = diag_login_completo.groupBy(
    "acao_anterior", "estado_entrada"
).agg(
    F.count("*").alias("n_contexto"),
    F.countDistinct("cd_bv").alias("n_clientes_contexto"),
)
diag_login_contexto = (
    diag_login_contexto.join(
        diag_login_totais, ["acao_anterior", "estado_entrada"]
    )
    .withColumn("p_proxima_dado_anterior",
                F.col("n_triplas") / F.col("n_contexto"))
    .withColumn("rank_contexto", F.row_number().over(
        Window.partitionBy("acao_anterior", "estado_entrada").orderBy(
            F.desc("n_triplas"), "acao_seguinte"
        )
    ))
)
diag_mostrar("06_login_destinos_por_contexto", diag_login_contexto.filter(
    (F.col("n_clientes_contexto") >= DIAG_CFG["min_clientes_contexto"])
    & (F.col("rank_contexto") <= 3)
).orderBy(F.desc("n_contexto"), "acao_anterior", "rank_contexto"))

diag_mostrar("06_login_destino_populacional", (
    diag_login_completo.groupBy("estado_entrada", "acao_seguinte").agg(
        F.count("*").alias("n_triplas"),
        F.countDistinct("cd_bv").alias("n_clientes"),
    ).join(
        diag_login_completo.groupBy("estado_entrada").agg(
            F.count("*").alias("n_total")
        ), "estado_entrada",
    )
    .withColumn("p_populacional",
                diag_dividir(F.col("n_triplas"), F.col("n_total")))
    .orderBy(F.desc("n_triplas"))
))

## 7. Calendário, recorrência e exposição observada

Compara início/fim do mês, dia da semana e mês. O denominador é a soma dos
**clientes-dias com qualquer atividade registrada**, não todos os clientes
bancários. Preenche zero para ações foco ausentes em dias com atividade.

Isso não estima probabilidade de um cliente inativo voltar ao banco; para
isso é necessário um painel elegível cliente-dia com cobertura confirmada.
Dias 29/30/31 não ocorrem em todos os meses: usar só contagens brutas pode
induzir uma leitura errada de sazonalidade. Não usamos média aritmética dos
dias 31 e 1 como medida de hábito.

In [ ]:
diag_ativos_dia = diag_eventos.groupBy(
    F.to_date("ts_estado").alias("dia")
).agg(F.countDistinct("cd_bv").alias("n_clientes_ativos"))
diag_clientes_acao_dia = (
    diag_eventos.filter(F.col("acao").isin(ACOES_FOCO_DIAG))
    .groupBy(F.to_date("ts_estado").alias("dia"), "acao")
    .agg(F.countDistinct("cd_bv").alias("n_clientes_acao"))
)
diag_focos = spark.createDataFrame(
    [(a,) for a in ACOES_FOCO_DIAG], "acao string"
)
diag_calendario = (
    diag_ativos_dia.crossJoin(F.broadcast(diag_focos))
    .join(diag_clientes_acao_dia, ["dia", "acao"], "left")
    .fillna({"n_clientes_acao": 0})
    .withColumn("dia_mes", F.dayofmonth("dia"))
    .withColumn("dias_para_fim_mes", F.datediff(F.last_day("dia"), "dia"))
    .withColumn("dia_semana", F.dayofweek("dia"))
    .withColumn("mes", F.date_format("dia", "yyyy-MM"))
    .withColumn("faixa_mes", F.when(F.col("dia_mes") <= 3, "01_inicio_1a3")
                .when(F.col("dias_para_fim_mes") <= 2, "03_ultimos_3dias")
                .otherwise("02_meio_mes"))
)
for dimensao in ["faixa_mes", "dia_semana", "mes"]:
    diag_mostrar(f"07_calendario_{dimensao}", (
        diag_calendario.groupBy("acao", dimensao).agg(
            F.countDistinct("dia").alias("n_datas_com_atividade"),
            F.sum("n_clientes_ativos").alias("n_clientes_dia_ativos"),
            F.sum("n_clientes_acao").alias("n_clientes_dia_com_acao"),
        ).withColumn("taxa_entre_clientes_dia_ativos", diag_dividir(
            F.col("n_clientes_dia_com_acao"), F.col("n_clientes_dia_ativos")
        )).orderBy("acao", dimensao)
    ))

diag_datas_acao = (
    diag_eventos.filter(F.col("acao").isin(ACOES_FOCO_DIAG))
    .select("cd_bv", "acao", F.to_date("ts_estado").alias("dia"))
    .distinct()
)
w_recorrencia = Window.partitionBy("cd_bv", "acao").orderBy("dia")
diag_recorrencia = diag_datas_acao.withColumn(
    "intervalo_dias", F.datediff("dia", F.lag("dia").over(w_recorrencia))
)
diag_mostrar("07_recorrencia_dias_distintos", (
    diag_recorrencia.groupBy("acao").agg(
    F.countDistinct("cd_bv").alias("n_clientes_com_acao"),
    F.count("intervalo_dias").alias("n_intervalos_observados"),
    F.expr("percentile_approx(intervalo_dias, array(0.25, 0.5, 0.75, 0.9))")
    .alias("quantis_intervalo_dias"),
)))
diag_rec_cliente = diag_datas_acao.groupBy("cd_bv", "acao").agg(
    F.countDistinct(F.date_format("dia", "yyyy-MM")).alias("n_meses_com_acao"),
    F.count("*").alias("n_dias_com_acao"),
)
diag_mostrar("07_recorrencia_meses", diag_rec_cliente.groupBy(
    "acao", "n_meses_com_acao"
).count().orderBy("acao", "n_meses_com_acao"))

## 8. Suporte no treino versus amostra efetiva do ajuste

Lê os contadores persistidos dos modelos. Não recria o `sampleBy` nem assume
que seria a mesma amostra. Ausência do modelo e alerta de limite paramétrico
são apresentados separadamente. Os clientes com muito histórico podem pesar
mais dentro da amostra de cada origem, mesmo com limite por origem.

In [ ]:
if spark.catalog.tableExists(DIAG_CFG["tabela_modelos"]):
    diag_modelos, DIAG_ID_MODELOS = diag_execucao_unica(
        diag_ler(DIAG_CFG["tabela_modelos"]),
        DIAG_CFG["id_execucao_modelos"], "modelos",
    )
    if DIAG_ID_BASE and DIAG_ID_MODELOS and DIAG_ID_BASE != DIAG_ID_MODELOS:
        raise ValueError("Base e modelos são de execuções diferentes.")
    diag_exigir_colunas(
        diag_modelos, {"origem", "status_modelo", "n_amostra"}, "modelos"
    )
    modelos_repetidos = (
        diag_modelos.groupBy("origem").count().filter("count > 1")
    )
    if diag_tem_linhas(modelos_repetidos):
        raise ValueError("Mais de um modelo por origem nessa execução.")
    col_modelos = ["origem", "status_modelo", "n_amostra"]
    extras = [
        "n_clientes", "n_eventos", "n_grupos", "alerta_limite", "detalhe"
    ]
    col_modelos += [c for c in extras if c in diag_modelos.columns]
    candidatas = diag_base_full.filter(
        ~F.col("validacao_cliente") & F.col("elegivel_ajuste")
    ).groupBy("estado").agg(
        F.count("*").alias("n_linhas_candidatas_ajuste"),
        F.countDistinct("cd_bv").alias("n_clientes_candidatos"),
    )
    diag_suporte = candidatas.join(
        diag_modelos.select(*col_modelos),
        F.col("estado") == F.col("origem"), "left",
    ).withColumn("fracao_amostrada_origem", diag_dividir(
        F.col("n_amostra"), F.col("n_linhas_candidatas_ajuste")
    ))
    diag_mostrar("08_suporte_modelos", diag_suporte.orderBy(
        F.desc("n_linhas_candidatas_ajuste")
    ))
    diag_mostrar("08_impacto_sem_ajuste", diag_suporte.filter(
        F.col("status_modelo").isNull()
        | (F.col("status_modelo") != "AJUSTADO")
    ).orderBy(F.desc("n_linhas_candidatas_ajuste")))
else:
    print("08: tabela de modelos ausente; seção não executada.")

## 9. Opcional — fonte bruta, profundidade e efeito da preparação

Ative `analisar_raw=True` e informe `inicio_raw` com o início real da extração.
O corte final usa `ts_corte_estado` persistido, quando único, salvo override.
Lê somente o período e os clientes selecionados na base preparada. Ainda
pode haver leitura substancial da tabela, dependendo do particionamento.

**Atenção:** profundidade normalizada dentro de um funil não demonstra ordem
cronológica entre funis diferentes. O diagnóstico separa colisões internas
de colisões entre ações. Debounce é uma heurística de telemetria: curta
duração não prova que duas operações são a mesma transação.

In [ ]:
if DIAG_CFG["analisar_raw"]:
    if not DIAG_CFG["inicio_raw"]:
        raise ValueError("Informe inicio_raw: início real da preparação.")
    fim_raw_diag = DIAG_CFG["fim_raw_exclusivo"]
    if fim_raw_diag is None:
        diag_exigir_colunas(
            diag_base_full, {"ts_corte_estado"}, "base preparada"
        )
        cortes = diag_base_full.select(F.date_format(
            "ts_corte_estado", "yyyy-MM-dd HH:mm:ss.SSSSSS"
        ).alias("corte")).distinct().limit(2).collect()
        if len(cortes) != 1 or cortes[0]["corte"] is None:
            raise ValueError("Informe fim_raw_exclusivo: corte não é único.")
        fim_raw_diag = cortes[0]["corte"]
    cli_raw_diag = diag_base.select(F.col("cd_bv").cast("string")).distinct()
    raw_diag = diag_ler(DIAG_CFG["fonte_raw"])
    diag_exigir_colunas(raw_diag, {
        "cd_bv", "dm_navegacao", "estado", "profundidade_max"
    }, "fonte raw")
    if raw_diag.schema["dm_navegacao"].dataType.simpleString() != "timestamp":
        raise ValueError("Fonte raw exige dm_navegacao timestamp.")
    periodo_diag = spark.createDataFrame(
        [(DIAG_CFG["inicio_raw"], str(fim_raw_diag))],
        "inicio string, fim string"
    ).select(
        F.to_timestamp("inicio").alias("inicio"),
        F.to_timestamp("fim").alias("fim"),
    ).first()
    if (periodo_diag["inicio"] is None or periodo_diag["fim"] is None
            or periodo_diag["inicio"] >= periodo_diag["fim"]):
        raise ValueError("Intervalo raw inválido.")
    raw_diag = (
        raw_diag.select(
            F.col("cd_bv").cast("string"),
            F.col("dm_navegacao").alias("ts_raw"), "estado",
            F.col("profundidade_max").cast("double").alias("prof"),
        ).filter(
            (F.col("ts_raw") >= F.lit(
                DIAG_CFG["inicio_raw"]
            ).cast("timestamp"))
            & (F.col("ts_raw") < F.lit(fim_raw_diag).cast("timestamp"))
        ).join(cli_raw_diag, "cd_bv", "left_semi")
        .join(diag_config, "estado", "left")
        .withColumn("acao", F.coalesce("funil", "estado"))
    )
    # null timestamps não podem ser atribuídos ao período; não os inferimos.
    diag_mostrar("09_raw_volume", raw_diag.agg(
        F.count("*").alias("n_registros_raw_no_periodo"),
        F.countDistinct("cd_bv").alias("n_clientes_com_raw"),
        F.sum(F.col("estado").isNull().cast("long")).alias("n_estado_nulo"),
        F.sum(F.col("prof").isNull().cast("long")).alias("n_prof_nula"),
    ))
    por_inst_estado = raw_diag.groupBy(
        "cd_bv", "ts_raw", "estado", "acao"
    ).agg(
        F.count("*").alias("n_registros"), F.max("prof").alias("prof")
    )
    w_inst_diag = Window.partitionBy("cd_bv", "ts_raw")
    por_inst_estado = por_inst_estado.withColumn(
        "max_prof", F.max("prof").over(w_inst_diag)
    )
    inst_raw_diag = por_inst_estado.groupBy("cd_bv", "ts_raw").agg(
        F.count("*").alias("n_estados"),
        F.countDistinct("acao").alias("n_acoes_distintas"),
        F.sum((F.col("prof") == F.col("max_prof")).cast("long"))
        .alias("n_estados_na_maior_prof"),
        F.sum(F.col("prof").isNull().cast("long")).alias("n_prof_ausentes"),
        F.sort_array(F.collect_set("acao")).alias("acoes_simultaneas"),
        F.sum("n_registros").alias("n_registros_raw"),
    ).withColumn("tipo_colisao", F.when(
        F.col("n_estados") <= 1, "ESTADO_UNICO"
    ).when(F.col("n_acoes_distintas") <= 1, "MESMO_FUNIL")
        .otherwise("FUNIS_OU_ACOES_DIFERENTES"))
    diag_mostrar("09_colisoes_profundidade", inst_raw_diag.groupBy(
        "tipo_colisao", "n_estados_na_maior_prof"
    ).agg(F.count("*").alias("n_instantes"),
          F.countDistinct("cd_bv").alias("n_clientes")))
    diag_mostrar("09_combinacoes_entre_funis", inst_raw_diag.filter(
        F.col("tipo_colisao") == "FUNIS_OU_ACOES_DIFERENTES"
    ).groupBy("acoes_simultaneas").count().orderBy(F.desc("count")))

    etapas_diag = raw_diag.groupBy("acao").agg(F.count("*").alias("n_raw"))
    etapas_diag = etapas_diag.join(
        diag_eventos.groupBy("acao").agg(F.count("*").alias("n_preparados")),
        "acao", "full",
    ).join(
        diag_blocos.groupBy("acao").agg(F.count("*").alias("n_blocos")),
        "acao", "full",
    ).fillna({"n_raw": 0, "n_preparados": 0, "n_blocos": 0})
    diag_mostrar("09_raw_preparado_blocos", etapas_diag.withColumn(
        "razao_preparado_raw",
        diag_dividir(F.col("n_preparados"), F.col("n_raw"))
    ).orderBy(F.desc("n_raw")))
    print("Diferenças de contagem não provam duplicatas: verificar linhagem.")
else:
    print("09 desativada. Ative analisar_raw após conferir período e custo.")

## 10. Scoring — concentração da saída e separação entre posições

O bloco usa uma amostra determinística por cliente da tabela LONG. O fator
é somente deste diagnóstico; o scoring que já existe não é alterado.

Top 1 não é comparável diretamente à prevalência histórica. Um mesmo Top 1
para muitos clientes ainda pode ter scores diferentes. Medimos também a
margem entre posições 1 e 2 e a repetição da lista de cinco ações.

Se houver vários snapshots por cliente, interrompe: selecione antes o recorte
correto. Ausência de `ts_corte_eventos` impede avaliar o recorte temporal.

In [ ]:
if DIAG_CFG["analisar_scoring"] and spark.catalog.tableExists(
    DIAG_CFG["tabela_scoring"]
):
    diag_score = diag_ler(DIAG_CFG["tabela_scoring"])
    diag_exigir_colunas(diag_score, {
        "cd_bv", "ranking_actionable", "actionable_acao", "actionable_score",
        "status_previsao", "ts_corte_eventos",
    }, "scoring LONG")
    diag_score = diag_score.filter(F.pmod(
        F.xxhash64(
            F.col("cd_bv").cast("string"), F.lit(DIAG_CFG["sal_scoring"])
        ),
        F.lit(DIAG_CFG["modulo_scoring"]),
    ) == 0)
    if diag_tem_linhas(diag_score.groupBy(
        "cd_bv", "ranking_actionable"
    ).count().filter("count > 1")):
        raise ValueError("Scoring com cliente/rank repetido: filtre snapshot.")
    if diag_score.select("ts_corte_eventos").distinct().limit(2).count() > 1:
        raise ValueError("Vários cortes no scoring: selecione um recorte.")
    diag_mostrar("10_scoring_cobertura", diag_score.agg(
        F.countDistinct("cd_bv").alias("n_clientes_amostra_diag"),
        F.countDistinct(F.when(
            F.col("ranking_actionable") == 1, F.col("cd_bv")
        ))
        .alias("n_clientes_com_rank"),
        F.min("ts_corte_eventos").alias("ts_corte_eventos"),
    ))
    diag_score_rank = diag_score.filter(
        F.col("ranking_actionable").between(1, 5)
    )
    diag_mostrar("10_scoring_top1", diag_score_rank.filter(
        F.col("ranking_actionable") == 1
    ).groupBy("actionable_acao").agg(
        F.count("*").alias("n_clientes_top1"),
        F.avg("actionable_score").alias("score_medio"),
        F.expr("percentile_approx(actionable_score, array(0.1, 0.5, 0.9))")
        .alias("quantis_score_top1"),
    ).orderBy(F.desc("n_clientes_top1")))
    diag_rank_clientes = diag_score_rank.groupBy("cd_bv").agg(
        F.max(F.when(
            F.col("ranking_actionable") == 1, F.col("actionable_score")
        ))
        .alias("score_1"),
        F.max(F.when(
            F.col("ranking_actionable") == 2, F.col("actionable_score")
        ))
        .alias("score_2"),
        F.sum("actionable_score").alias("massa_top5"),
        F.to_json(F.sort_array(F.collect_list(F.struct(
            "ranking_actionable", "actionable_acao"
        )))).alias("assinatura_acoes_top5"),
    ).withColumn("margem_1_2", F.col("score_1") - F.col("score_2"))
    diag_mostrar("10_scoring_margem", diag_rank_clientes.agg(
        F.count("*").alias("n_clientes"),
        F.expr("percentile_approx(margem_1_2, array(0.1, 0.5, 0.9))")
        .alias("quantis_margem"),
        F.countDistinct("assinatura_acoes_top5").alias("n_listas_distintas"),
        F.avg("massa_top5").alias("massa_media_top5"),
    ))
    diag_mostrar("10_scoring_listas_repetidas", diag_rank_clientes.groupBy(
        "assinatura_acoes_top5"
    ).count().orderBy(F.desc("count")))
else:
    print("10: scoring desativado ou tabela ausente.")

## 11. Consulta opcional por cliente e encerramento

Os relatórios automáticos acima são agregados, sem listar IDs. A função abaixo
serve para inspeção interna de um cliente presente na **amostra histórica
analisada**. Ausência nessa amostra não significa ausência de histórico no banco.

`DIAG_RESULTADOS` guarda DataFrames dos relatórios para exploração local.
Nada é salvo em Delta por este notebook.

In [ ]:
def diag_consultar_cliente(cd_bv: str) -> None:
    identificador = str(cd_bv).strip()
    if not identificador:
        raise ValueError("Informe um cd_bv não vazio.")
    perfil = diag_perfil_clientes.filter(
        F.col("cd_bv") == F.lit(identificador)
    )
    if not diag_tem_linhas(perfil):
        print("Cliente não encontrado no escopo histórico analisado.")
        return
    perfil.show(truncate=False)
    diag_freq_cliente.filter(F.col("cd_bv") == F.lit(identificador)).select(
        "acao", F.col("count").alias("n_blocos"),
        F.col("p").alias("participacao_cliente"),
    ).orderBy(F.desc("n_blocos")).show(30, truncate=False)


def diag_liberar_cache() -> None:
    for df in DIAG_CACHES:
        df.unpersist()
    DIAG_CACHES.clear()


# Exemplo de uso interno:
# diag_consultar_cliente("SEU_CD_BV")
# Depois de terminar a exploração:
# diag_liberar_cache()

print("Diagnóstico concluído. Nenhuma tabela ou modelo foi modificado.")
print("Relatórios disponíveis:", sorted(DIAG_RESULTADOS))

## Interpretação e decisão

| Achado | Interpretação a investigar | Próxima ação |
|---|---|---|
| Muito volume bruto, poucos clientes/blocos | Telemetria ou poucos usuários muito ativos | Auditar eventos e granularidade |
| Alta presença, alta diversidade individual, Top 1 uniforme | Saída perde contexto ou candidatas muito restritas | Testar memória/calendário/histórico |
| Após login, destinos variam com ação anterior | Evidência exploratória de dependência do caminho | Comparar modelo contextual fora do tempo |
| Boleto raro também por cliente | Raridade real, não necessariamente falha | Avaliar capacidade operacional e métricas por ação |
| Muitas colisões entre funis | Maior profundidade pode descartar outra ação | Revisar regra com origem/event_id |
| Base preparada ampla, amostra por origem pequena | Suporte efetivo difere do catálogo | Planejar amostragem sem distorcer probabilidades |
| Padrões de calendário só em contagens brutas | Efeito de exposição ou volume | Usar taxas e separar meses/perfis |
| Histórias curtas concentram em uma ação | Pouca oportunidade de observar diversidade | Separar cobertura de comportamento |

### Limitações deliberadas

- Não prova que eventos são transações reais. Sem identificador da operação,
  timestamps e debounce não permitem essa conclusão.
- A preparação remove etapas simultâneas e agrupa repetições. A coluna `n_ev`
  pode contar registros associados, inclusive candidatos descartados por
  profundidade; não é um contador certificado de transações.
- As triplas e transições usam saídas observadas, sem tratar censurados como
  negativos. Há seleção pela disponibilidade de futuro.
- O debounce usa o último timestamp do bloco. Em backtests de snapshots,
  a preparação precisa ser refeita somente com eventos anteriores ao corte;
  um bloco encerrado usando o futuro não pode definir a informação em t0.
- Não estima risco de retorno para clientes sem atividade: falta painel
  de elegibilidade e cobertura por cliente-dia.
- Não mede impacto incremental de mensagem, campanha ou recomendação.
- Não é uma nova validação final. Reutilizar repetidamente o mesmo holdout
  para decidir alterações transforma-o em conjunto de desenvolvimento.
- A classificação macro-aware e a whitelist mudam o alvo de primeira
  passagem. A validação futura deve usar exatamente a mesma definição.

### Referências técnicas

- Fonte do projeto: `nba_v22_01_base_ultima_acao_debounce.py`;
  `nba_v22_02_modelo_semimarkov.py`;
  `nba_v222_04b_macroaware_actionable_long.py`.
- Spark: https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.toPandas.html
- Spark, amostragem: https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrameStatFunctions.sampleBy.html
- Validação: https://scikit-learn.org/stable/modules/cross_validation.html
- Memória em trajetórias: Rosvall et al., https://arxiv.org/abs/1305.4807
- Personalização: Rendle et al., https://doi.org/10.1145/1772690.1772773